In [ ]:
import math
import random
from dataclasses import dataclass, field, fields
from itertools import count
from statistics import mean
from typing import Callable, Collection, Generic, Sequence, TypeVar, cast

import matplotlib.pyplot as plt

import data.cmudict as cmudict
from state_merging.algorithms.ostia import ostia
from state_merging.automata.SFST import SFST
from state_merging.operations.learner import MergeResults

T = TypeVar('T')
T_ = TypeVar('T_')

random.seed(42)

In [ ]:
@dataclass
class Results(Generic[T]):
    sample_count: T = field()
    sample_input_size: T = field()
    ptt_state_count: T = field()
    merged_state_count: T = field()
    proportion_states_merged: T = field()

    @classmethod
    def from_merge_results(cls, merge_results: MergeResults) -> 'Results[float]':
        return Results(
            sample_count=merge_results.sample_count,
            sample_input_size=merge_results.sample_input_size,
            ptt_state_count=merge_results.ptt_state_count,
            merged_state_count=merge_results.merged_state_count,
            proportion_states_merged=1 - merge_results.merged_state_count/merge_results.ptt_state_count
        )

    @classmethod
    def map(cls, fun: Callable[[T], T_], results: 'Results[T]') -> 'Results[T_]':
        return cls(**{
            f.name: fun(getattr(results, f.name))
            for f in fields(cls)
        }) # type: ignore

    @classmethod
    def collect(cls, all_results: Collection['Results[T]']) -> 'Results[list[T]]':
        return cls(**{
            f.name: [getattr(results, f.name) for results in all_results]
            for f in fields(cls)
        }) # type: ignore

    @classmethod
    def mean(cls, all_results: Collection['Results[float]']) -> 'Results[float]':
        return Results.map(mean, Results.collect(all_results)) # type: ignore

In [ ]:
input_set: set[str] = cmudict.make_input_set()
output_set: set[str] = cmudict.make_output_set()
dataset: list[tuple[str, list[str]]] = list(cmudict.make_deterministic_sample())

print(
    f"loaded {len(dataset)} samples from the CMU Pronouncing Dictionary, e.g.:",
    random.sample(dataset, k=1)[0]
)

print(f"input alphabet:", input_set)
print(f"output alphabet:", output_set)

In [ ]:
iterations: int = 3
target_sample_sizes: list[int] = [int(10 ** (i / 4 + 2)) for i in range(7)]

print(
    f"to run {iterations} iterations of OSTIA with randomized merge order",
    f"on random samples of size {target_sample_sizes}"
)

In [ ]:
mean_results_per_config: list[Results[float]] = []

for k in target_sample_sizes:
    print(f"running with sample size {k}")

    results_per_iter: list[Results[float]] = []

    for i in range(iterations):
        print(i, end=' ')

        fst: SFST[int, str, Sequence[str]]
        fst, results = ostia(
            input_set=input_set,
            dataset=random.sample(dataset, k=k),
            epsilon=[],
            concat=lambda v1, v2: cast(list[str], v1) + cast(list[str], v2),
            choose_transition=lambda _, trs: random.choice(list(trs)),
            search_iter=lambda _, qs: random.sample(list(qs), k=len(qs)),
            state_supply=count()
        )

        results_per_iter.append(Results.from_merge_results(results))

    mean_results_per_config.append(Results.mean(results_per_iter))

    print()

print("done")

In [ ]:
r: Results[list[float]] = Results.collect(mean_results_per_config) # type: ignore

alpha = 0.3
y_range_pad = 1.1

In [ ]:
# type: ignore

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

y_log_datasets = [
    [math.log10(e) for e in r.sample_input_size],
    [math.log10(e) for e in r.ptt_state_count],
    [math.log10(e) for e in r.merged_state_count]
]

y_log_midpoints = [max(d)/2 + min(d)/2 for d in y_log_datasets]
y_log_ranges = [max(d) - min(d) for d in y_log_datasets]
y_log_range = max(y_log_ranges) * y_range_pad

axes[0].loglog(r.sample_count, r.sample_input_size, 'o', label='sample_input_size')
axes[0].set_title('Sum of Input Lengths vs Sample Size')
axes[0].set_xlabel('sample_count')
axes[0].set_ylabel('sample_input_size')

axes[1].loglog(r.sample_count, r.ptt_state_count, 'o', label='ppt_state_count')
axes[1].set_title('PTT State Count vs Sample Count')
axes[1].set_xlabel('sample_count')
axes[1].set_ylabel('ppt_state_count')

axes[2].loglog(r.sample_count, r.merged_state_count, 'o', label='merged_state_count')
axes[2].set_title('Learned FST State Count vs Sample Count')
axes[2].set_xlabel('sample_count')
axes[2].set_ylabel('merged_state_count')

for i in [0, 1, 2]:
    axes[i].set_ylim(
        10**(y_log_midpoints[i] - y_log_range/2),
        10**(y_log_midpoints[i] + y_log_range/2)
    )

    axes[i].grid(True, which='both', alpha=alpha)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# type: ignore

fig, axes = plt.subplots(1, 1, figsize=(4, 4))

axes.semilogx(r.sample_count, [100 * e for e in r.proportion_states_merged], 'o', label='100 * proportion_states_merged')
axes.set_title('Percent of States Merged vs Sample Count')
axes.set_xlabel('sample_count')
axes.set_ylabel('100 * proportion_states_merged')
axes.set_ylim(100 * (1 - 2 * (1 - mean(r.proportion_states_merged))), 100)
axes.grid(True, alpha=alpha)
axes.legend()

plt.tight_layout()
plt.show()